
# Seepage vs Δh/visitor_h — Global Bounds + n‑Weighted Grouped Summaries (SSE)

This notebook starts from the **updates** workflow and adds two things:

1. **Global offset bounds**: compute one pair of bounds per predictor (Δh, visitor_h), then use those bounds everywhere (pooled and grouped).
2. **Grouped summaries weighted by event size**: report **n‑weighted medians** of parameters across events (so pooled and grouped are more comparable).

**Models (SSE throughout):**
- Linear (Δh): \( S = K\,\Delta h \)
- Offset–Power (Δh): \( S = K\,\Delta h\,\Big(\dfrac{\Delta h+\text{offset}}{x_{\mathrm{ref}}}\Big)^{b} \)
- Offset–Power (visitor\_h): \( S = K\,\Delta h\,\Big(\dfrac{\text{visitor\_h}+\text{offset}}{x_{\mathrm{ref}}}\Big)^{b} \)

**Key choices**
- `x_ref` is the global median of the *relevant* predictor (Δh or visitor\_h).
- Loss is **SSE** only (matches updates notebook).
- Bounds are global, physically‑motivated: lower bound enforces \(x+\text{offset}\ge 0\), upper bound is a gentle site‑scale cap.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# ---- Paths ----
DATA_PATH = Path("../data/processed/pooled_event_df.csv")  # point to your CSV

# ---- Numerics ----
EPS = 1e-12

# ---- Site-scale caps for the upper bound (tunable but small) ----
SITE_CAP_DH  = 0.50   # maximum positive offset allowed for Δh
SITE_CAP_VIS = 2.00   # maximum positive offset allowed for visitor_h

print("Using data:", DATA_PATH)

Using data: ../data/processed/pooled_event_df.csv


In [2]:

# Load data; ensure numeric types
df = pd.read_csv(DATA_PATH)
df["delta_h"]  = pd.to_numeric(df.get("delta_h"), errors="coerce")
df["seepage"]  = pd.to_numeric(df.get("seepage"), errors="coerce")
if "visitor_h" in df.columns:
    df["visitor_h"] = pd.to_numeric(df.get("visitor_h"), errors="coerce")
else:
    df["visitor_h"] = np.nan
df["n_full"] = df.groupby("event")["event"].transform("size").astype("int64")

# Minimal clean
df = df.dropna(subset=["delta_h", "seepage"]).copy()
df = df.query("seepage <4 and n_full > 5")
# Event size (days)
df["n"] = df.groupby("event")["event"].transform("size").astype("int64")

print(f"Rows: {len(df):,} | Events: {df['event'].nunique():,}")
df.head(3)

Rows: 378 | Events: 24


,date,seepage,delta_h,waveHs,visitor_h,date.1,event,n,n_full
0,2014-09-20 00:00:00+00:00,0.240583,0.728999,0.422800,1.873488,2014-09-20,0,28,28
1,2014-09-21 00:00:00+00:00,0.887796,0.803515,0.345512,1.943640,2014-09-21,0,28,28
2,2014-09-22 00:00:00+00:00,0.896792,0.906064,0.324142,2.014919,2014-09-22,0,28,28


In [3]:

# Global x_ref = global medians
XREF_DH  = float(np.nanmedian(df["delta_h"].values))
XREF_VIS = float(np.nanmedian(df["visitor_h"].values)) if df["visitor_h"].notna().any() else np.nan

def global_offset_bounds(x_series, site_cap):
    # Compute global offset bounds for a predictor x:
    #   LB ensures x + offset >= EPS; UB is a gentle site cap tied to spread.
    x = x_series.to_numpy(float)
    if not np.isfinite(x).any():
        return 0.0, site_cap
    lb = -float(np.nanmin(x)) + 1e-9  # keep (x + offset) >= ~0
    # A gentle UB from spread (approx IQR) plus site cap
    q05, q95 = np.nanquantile(x, [0.05, 0.95])
    spread = float(q95 - q05)
    ub = min(0.30 * max(spread, 1e-6), site_cap)
    if ub <= lb + 1e-6:
        ub = lb + max(0.05 * max(spread, 1e-6), 0.05)  # ensure feasible range
    return float(lb), float(ub)

OFFSET_LB_DH,  OFFSET_UB_DH  = global_offset_bounds(df["delta_h"],  SITE_CAP_DH)
if df["visitor_h"].notna().any():
    OFFSET_LB_VIS, OFFSET_UB_VIS = global_offset_bounds(df["visitor_h"].dropna(), SITE_CAP_VIS)
else:
    OFFSET_LB_VIS, OFFSET_UB_VIS = (0.0, 0.0)  # unused

print("Global x_ref (Δh):", XREF_DH)
print("Global x_ref (visitor_h):", XREF_VIS)
print("Δh  offset bounds:", (OFFSET_LB_DH, OFFSET_UB_DH))
print("vis offset bounds:", (OFFSET_LB_VIS, OFFSET_UB_VIS))

Global x_ref (Δh): 1.677690125
Global x_ref (visitor_h): 2.684050524390244
Δh  offset bounds: (-0.6246971656666666, 0.42568045219696976)
vis offset bounds: (-1.6807179989999999, 0.4122834337499999)


In [4]:

def r2_centered(y, yhat):
    # Centered R^2 using the sample mean of y
    y = np.asarray(y, float); yhat = np.asarray(yhat, float)
    sst = np.sum((y - np.mean(y))**2)
    if sst <= 0:
        return np.nan
    sse = np.sum((y - yhat)**2)
    return 1.0 - sse / sst

def weighted_median(values, weights):
    # n-weighted median (by event-size) across events
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[ok], w[ok]
    if len(v) == 0:
        return np.nan
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = np.cumsum(w) / np.sum(w)
    return float(v[np.searchsorted(cw, 0.5)])

# Model forms
def model_linear(K, x):
    return K * x

def model_offset_power_delta_h(K, offset, b, x_dh):
    base = np.maximum(x_dh + offset, EPS)
    xref = max(XREF_DH, EPS)
    return K * x_dh * np.power(base / xref, b)

def model_offset_power_visitor_h(K, offset, b, x_dh, x_vis):
    base = np.maximum(x_vis + offset, EPS)
    xref = max(XREF_VIS, EPS)
    return K * x_dh * np.power(base / xref, b)

In [5]:

def pooled_linear_delta_h(df_in):
    x = df_in["delta_h"].to_numpy(float)
    y = df_in["seepage"].to_numpy(float)
    # Closed-form OLS through origin
    K = float(np.dot(x, y) / max(np.dot(x, x), EPS))
    K = max(K, 1e-12)
    yhat = model_linear(K, x)
    return {"K": K, "R2": r2_centered(y, yhat), "n": len(x)}

def pooled_offset_power_delta_h(df_in):
    x = df_in["delta_h"].to_numpy(float)
    y = df_in["seepage"].to_numpy(float)

    def objective(theta):
        K, off, b = theta
        if (off < OFFSET_LB_DH) or (off > OFFSET_UB_DH) or (K <= 0) or (b < 0):
            return 1e99
        yhat = model_offset_power_delta_h(K, off, b, x)
        return float(np.sum((y - yhat)**2))

    # Reasonable start: K from linear; offset mid‑range; b near 0.3–0.6
    K0 = float(np.dot(x, y) / max(np.dot(x, x), EPS))
    off0 = (OFFSET_LB_DH + OFFSET_UB_DH)/2
    b0 = 0.5
    res = minimize(objective, x0=[max(K0, 1e-12), off0, b0],
                   bounds=[(1e-12, None), (OFFSET_LB_DH, OFFSET_UB_DH), (1e-12, None)],
                   method="L-BFGS-B")
    K, off, b = map(float, res.x)
    yhat = model_offset_power_delta_h(K, off, b, x)
    return {"K": K, "offset": off, "b": b, "R2": r2_centered(y, yhat), "n": len(x)}

def pooled_offset_power_visitor_h(df_in):
    if not df_in["visitor_h"].notna().any():
        return {}
    sub = df_in.dropna(subset=["visitor_h"]).copy()
    x = sub["delta_h"].to_numpy(float)
    xv = sub["visitor_h"].to_numpy(float)
    y = sub["seepage"].to_numpy(float)

    def objective(theta):
        K, off, b = theta
        if (off < OFFSET_LB_VIS) or (off > OFFSET_UB_VIS) or (K <= 0) or (b < 0):
            return 1e99
        yhat = model_offset_power_visitor_h(K, off, b, x, xv)
        return float(np.sum((y - yhat)**2))

    K0 = float(np.dot(x, y) / max(np.dot(x, x), EPS))
    off0 = (OFFSET_LB_VIS + OFFSET_UB_VIS)/2
    b0 = 0.5
    res = minimize(objective, x0=[max(K0, 1e-12), off0, b0],
                   bounds=[(1e-12, None), (OFFSET_LB_VIS, OFFSET_UB_VIS), (1e-12, None)],
                   method="L-BFGS-B")
    K, off, b = map(float, res.x)
    yhat = model_offset_power_visitor_h(K, off, b, x, xv)
    return {"K": K, "offset": off, "b": b, "R2": r2_centered(y, yhat), "n": len(x)}

pooled_lin = pooled_linear_delta_h(df)
pooled_dh  = pooled_offset_power_delta_h(df)
pooled_vis = pooled_offset_power_visitor_h(df)

print("=== POOLED (global bounds) ===")
print(f"Linear (Δh):       K={pooled_lin['K']:.3f},   R²={pooled_lin['R2']:.3f}, n={pooled_lin['n']}")
print(f"Offset–Power (Δh): K={pooled_dh['K']:.3f}, off={pooled_dh['offset']:.3f}, b={pooled_dh['b']:.3f}, R²={pooled_dh['R2']:.3f}, n={pooled_dh['n']}")
if pooled_vis:
    print(f"Offset–Power (visitor_h): K={pooled_vis['K']:.3f}, off={pooled_vis['offset']:.3f}, b={pooled_vis['b']:.3f}, R²={pooled_vis['R2']:.3f}, n={pooled_vis['n']}")
else:
    print("Offset–Power (visitor_h): N/A (visitor_h missing)")

=== POOLED (global bounds) ===
Linear (Δh):       K=1.052,   R²=0.462, n=378
Offset–Power (Δh): K=0.947, off=0.426, b=0.353, R²=0.481, n=378
Offset–Power (visitor_h): K=1.204, off=-1.152, b=0.289, R²=0.485, n=378


In [6]:

def fit_grouped_sse(df_in, min_n=6):
    rows_lin = []
    rows_dh  = []
    rows_vis = []

    for e, g in df_in.groupby("event"):
        n = len(g)
        if n < min_n:
            continue
        x = g["delta_h"].to_numpy(float)
        y = g["seepage"].to_numpy(float)

        # Linear (Δh)
        K = float(np.dot(x, y) / max(np.dot(x, x), EPS))
        K = max(K, 1e-12)
        yhat = model_linear(K, x)
        rows_lin.append({"event": e, "n": n, "K_lin": K, "R2_lin": r2_centered(y, yhat)})

        # Offset–Power (Δh)
        def objective(theta):
            Kp, offp, bp = theta
            if (offp < OFFSET_LB_DH) or (offp > OFFSET_UB_DH) or (Kp <= 0) or (bp < 0):
                return 1e99
            yhatp = model_offset_power_delta_h(Kp, offp, bp, x)
            return float(np.sum((y - yhatp)**2))

        K0 = K
        off0 = (OFFSET_LB_DH + OFFSET_UB_DH)/2
        b0 = 0.5
        res = minimize(objective, x0=[K0, off0, b0],
                       bounds=[(1e-12, None), (OFFSET_LB_DH, OFFSET_UB_DH), (1e-12, None)],
                       method="L-BFGS-B")
        Kp, offp, bp = map(float, res.x)
        yhatp = model_offset_power_delta_h(Kp, offp, bp, x)
        rows_dh.append({"event": e, "n": n, "K_pow": Kp, "offset": offp, "b": bp,
                        "R2_pow": r2_centered(y, yhatp)})

        # Offset–Power (visitor_h), if available
        if df_in["visitor_h"].notna().any():
            xv = g["visitor_h"].to_numpy(float)
            xv_ok = np.isfinite(xv).sum() >= min_n
            if xv_ok:
                def objective_v(theta):
                    Kv, offv, bv = theta
                    if (offv < OFFSET_LB_VIS) or (offv > OFFSET_UB_VIS) or (Kv <= 0) or (bv < 0):
                        return 1e99
                    yhatv = model_offset_power_visitor_h(Kv, offv, bv, x, xv)
                    return float(np.sum((y - yhatv)**2))

                Kv0 = K
                offv0 = (OFFSET_LB_VIS + OFFSET_UB_VIS)/2
                bv0 = 0.5
                resv = minimize(objective_v, x0=[Kv0, offv0, bv0],
                                bounds=[(1e-12, None), (OFFSET_LB_VIS, OFFSET_UB_VIS), (1e-12, None)],
                                method="L-BFGS-B")
                Kv, offv, bv = map(float, resv.x)
                yhatv = model_offset_power_visitor_h(Kv, offv, bv, x, xv)
                rows_vis.append({"event": e, "n": n, "K_pow": Kv, "offset": offv, "b": bv,
                                 "R2_pow": r2_centered(y, yhatv)})

    ev_lin = pd.DataFrame(rows_lin).sort_values("event").reset_index(drop=True)
    ev_dh  = pd.DataFrame(rows_dh).sort_values("event").reset_index(drop=True)
    ev_vis = pd.DataFrame(rows_vis).sort_values("event").reset_index(drop=True) if rows_vis else pd.DataFrame()

    # n-weighted medians
    def wmed(ev_df, label, include_linear=False):
        if ev_df.empty: return pd.DataFrame()
        w = ev_df["n"].to_numpy(float)
        items = []
        if include_linear:
            items.append(("K (linear, w‑med)", weighted_median(ev_lin["K_lin"], ev_lin["n"])))
        if "K_pow" in ev_df.columns:
            items.extend([
                ("K (offset‑power, w‑med)", weighted_median(ev_df["K_pow"], w)),
                ("offset (w‑med)",          weighted_median(ev_df["offset"], w)),
                ("b (w‑med)",               weighted_median(ev_df["b"], w)),
            ])
        return pd.DataFrame([dict(items)], index=[label])

    summary_lin = wmed(ev_lin, "Grouped Δh (linear)", include_linear=True)
    summary_dh  = wmed(ev_dh,  "Grouped Δh (offset‑power)")
    summary_vis = wmed(ev_vis, "Grouped visitor_h (offset‑power)") if not ev_vis.empty else pd.DataFrame()

    return ev_lin, ev_dh, ev_vis, summary_lin, summary_dh, summary_vis

ev_lin, ev_dh, ev_vis, summ_lin, summ_dh, summ_vis = fit_grouped_sse(df, min_n=6)

print("\n=== GROUPED — n‑weighted medians (global bounds) ===")
if not summ_lin.empty:
    print(summ_lin.round(3).to_string(index=True))
if not summ_dh.empty:
    print(summ_dh.round(3).to_string(index=True))
if not summ_vis.empty:
    print(summ_vis.round(3).to_string(index=True))


=== GROUPED — n‑weighted medians (global bounds) ===
                     K (linear, w‑med)
Grouped Δh (linear)              1.014
                           K (offset‑power, w‑med)  offset (w‑med)  b (w‑med)
Grouped Δh (offset‑power)                    0.897             0.0      0.227
                                  K (offset‑power, w‑med)  offset (w‑med)  b (w‑med)
Grouped visitor_h (offset‑power)                    1.056          -0.514       0.32


In [7]:

rows = []

# Linear (Δh)
rows.append({
    "Model": "Linear (Δh)",
    "Pooled K": pooled_lin["K"],
    "Grouped K (w‑med)": float(summ_lin["K (linear, w‑med)"].iloc[0]) if not summ_lin.empty else np.nan,
})

# Offset–Power (Δh)
rows.append({
    "Model": "Offset–Power (Δh)",
    "Pooled K": pooled_dh["K"], "Pooled offset": pooled_dh["offset"], "Pooled b": pooled_dh["b"],
    "Grouped K (w‑med)": float(summ_dh["K (offset‑power, w‑med)"].iloc[0]) if not summ_dh.empty else np.nan,
    "Grouped offset (w‑med)": float(summ_dh["offset (w‑med)"].iloc[0]) if not summ_dh.empty else np.nan,
    "Grouped b (w‑med)": float(summ_dh["b (w‑med)"].iloc[0]) if not summ_dh.empty else np.nan,
})

# Offset–Power (visitor_h), if present
if pooled_vis:
    rows.append({
        "Model": "Offset–Power (visitor_h)",
        "Pooled K": pooled_vis["K"], "Pooled offset": pooled_vis["offset"], "Pooled b": pooled_vis["b"],
        "Grouped K (w‑med)": float(summ_vis["K (offset‑power, w‑med)"].iloc[0]) if not summ_vis.empty else np.nan,
        "Grouped offset (w‑med)": float(summ_vis["offset (w‑med)"].iloc[0]) if not summ_vis.empty else np.nan,
        "Grouped b (w‑med)": float(summ_vis["b (w‑med)"].iloc[0]) if not summ_vis.empty else np.nan,
    })

comp = pd.DataFrame(rows)
num_cols = comp.select_dtypes(include=["float","int"]).columns
comp[num_cols] = comp[num_cols].round(3)
comp[num_cols] = comp[num_cols].mask(np.isclose(comp[num_cols], 0, atol=5e-4), 0.0)

print("\n=== COMPARISON (Pooled vs Grouped w‑med; global bounds) ===")
print(comp.to_string(index=False))

# Save for convenience
# out_csv = Path("/mnt/data/pooled_vs_grouped_global_bounds_weighted_medians.csv")
# comp.to_csv(out_csv, index=False)
# print("\nSaved comparison CSV:", out_csv)


=== COMPARISON (Pooled vs Grouped w‑med; global bounds) ===
                   Model  Pooled K  Grouped K (w‑med)  Pooled offset  Pooled b  Grouped offset (w‑med)  Grouped b (w‑med)
             Linear (Δh)     1.052              1.014            NaN       NaN                     NaN                NaN
       Offset–Power (Δh)     0.947              0.897          0.426     0.353                   0.000              0.227
Offset–Power (visitor_h)     1.204              1.056         -1.152     0.289                  -0.514              0.320


In [8]:
comp

,Model,Pooled K,Grouped K (w‑med),Pooled offset,Pooled b,Grouped offset (w‑med),Grouped b (w‑med)
0,Linear (Δh),1.052,1.014,NaN,NaN,NaN,NaN
1,Offset–Power (Δh),0.947,0.897,0.426,0.353,0.000,0.227
2,Offset–Power (visitor_h),1.204,1.056,-1.152,0.289,-0.514,0.320


In [9]:
!open .